# OnePilot - Architecture RAG Complete
## Du RAG classique au Precision Agent - Benchmark evolutif

Ce notebook mesure et compare TOUS les types de RAG implementes dans OnePilot :

| Sprint | Composant | Role |
|--------|-----------|------|
| S7A | Hybrid RAG (BM25 + pgvector + RRF) | Retrieval enrichi |
| S7B | GraphRAG BFS | Exploration FK |
| S7C | Corrective RAG (CRAG) | Qualite contexte |
| S8 | Agentic RAG - ReAct | Boucle autonome |
| S8.5 | Direct SQL Bypass | 0 LLM |
| S8.6 | Validation Option C | Couverture entites |
| S9 | Multi-Agent RAG + Orchestrateur | Routing intelligent |
| S10 | Agent Precision | Schema structure reel |


In [ ]:
import requests, time, re, json, unicodedata
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display, HTML

BASE_URL  = 'http://onepilot_api:8000'
SOURCE_ID = '03add1dc-754a-476a-bbd5-1e53a05bf8d7'

try:
    r = requests.get(f'{BASE_URL}/health', timeout=30)
    print(f'API connectee - status={r.status_code}')
except Exception as e:
    print(f'Connexion echouee : {e}')


In [ ]:
# ============================================================
# UTILITAIRES COMMUNS
# ============================================================

HALLUCINATION_TABLES = [
    'ligne_de_financement','di_finline_dtls','di_fnline','di_trnb',
    'di_tl_prppmt','financement_table','banque_table',
    'gs_blngst_comp','rc_b2gsc_2_bnksttp','gs_bnkgrp_2_bnk',
]

def _norm(s):
    import unicodedata as ud
    s = ud.normalize('NFD', s.lower())
    return ''.join(c for c in s if ud.category(c) != 'Mn')

def detect_hallucination(sql):
    if not sql: return True, ['SQL vide']
    sn = _norm(sql)
    found = [t for t in HALLUCINATION_TABLES if _norm(t) in sn]
    return len(found) > 0, found

def validate_coverage(question, sql):
    if not sql: return False, 0.0, ['SQL vide']
    entities = {}
    years = re.findall(r'\\b(?:19|20)\\d{2}\\b', question)
    if years: entities['years'] = years
    banks = ['bnp','banque postale','societe generale','stb','biat']
    for b in banks:
        if b in _norm(question): entities.setdefault('banks',[]).append(b)
    currencies = re.findall(r'\\b(TND|EUR|USD)\\b', question, re.IGNORECASE)
    if currencies: entities['currencies'] = [c.upper() for c in currencies]
    if not entities: return True, 1.0, []
    sn = _norm(sql); missing, total, covered = [], 0, 0
    for etype, vals in entities.items():
        for v in vals:
            total += 1
            if _norm(v) in sn: covered += 1
            else: missing.append(f'{etype}:{v}')
    score = covered / total if total > 0 else 1.0
    return score >= 0.6, round(score, 2), missing

def call_api(question, endpoint='/orchestrator/query'):
    t0 = time.time()
    try:
        r = requests.post(f'{BASE_URL}{endpoint}',
            json={'question': question, 'source_id': SOURCE_ID, 'dialect': 'mssql'},
            timeout=180)
        elapsed = round((time.time()-t0)*1000)
        data = r.json()
        sql  = data.get('sql','')
        agent= data.get('agent_type','unknown')
        iters= data.get('iterations',1)
        method=data.get('method','unknown')
        is_hall, hall_t = detect_hallucination(sql)
        cov_ok, cov_sc, missing = validate_coverage(question, sql)
        success = bool(sql and not is_hall and cov_ok and 'ERROR' not in sql.upper())
        return {'question':question,'sql':sql,'agent':agent,'method':method,
                'iterations':iters,'elapsed_ms':elapsed,'success':success,
                'cov_score':cov_sc,'missing':missing,
                'is_hallucination':is_hall,'halluc_tables':hall_t}
    except Exception as e:
        elapsed = round((time.time()-t0)*1000)
        return {'question':question,'sql':'','agent':'error','method':'error',
                'iterations':0,'elapsed_ms':elapsed,'success':False,
                'cov_score':0.0,'missing':[str(e)],
                'is_hallucination':True,'halluc_tables':[str(e)]}

def print_result(r):
    agent_icon = {
        'direct_sql' :'[Direct   ]',
        'precision'  :'[Precision]',
        'react_plus' :'[ReAct+   ]',
        'multi_query':'[Multi    ]',
        'unknown'    :'[Unknown  ]',
    }.get(r['agent'],'[?        ]')
    ok   = 'OK  ' if r['success'] else 'FAIL'
    hall = f' HALL:{r["halluc_tables"][:1]}' if r['is_hallucination'] else ''
    cov  = f' cov={r["cov_score"]:.0%}' if r['missing'] else ' cov=OK'
    print(f'  [{ok}] {agent_icon} {r["elapsed_ms"]:>6}ms{cov}{hall}  {r["question"][:55]}')

print('Utilitaires OK')


In [ ]:
# ============================================================
# COUCHE 1 : RAG CLASSIQUE (Sprint 7 - baseline)
# Test : questions simples sans filtre -> schema retrieve -> LLM
# Composants actifs : NLU -> CRAG -> LLM
# ============================================================

print('='*65)
print('  COUCHE 1 - RAG Classique (S7 baseline)')
print('  NLU -> Retrieval -> LLM -> SQL')
print('='*65)

QUESTIONS_RAG_CLASSIQUE = [
    'liste les tables disponibles',
    'quels systemes SXA existent',
    'liste les groupes de societes',
    'quels sont les types de transactions',
    'liste les parametres systeme',
]

results_rag = []
for q in QUESTIONS_RAG_CLASSIQUE:
    r = call_api(q)
    results_rag.append(r)
    print_result(r)

ok_rag = [r for r in results_rag if r['success']]
avg_rag = round(sum(r['elapsed_ms'] for r in results_rag)/len(results_rag))
print(f'\n  Succes={len(ok_rag)}/{len(results_rag)} | Moy={avg_rag}ms')
print('  Composants : BM25 + pgvector + RRF + Reranker + LLM')


In [ ]:
# ============================================================
# COUCHE 2 : HYBRID RAG (Sprint 7A)
# Test : questions avec termes metier -> BM25 capte noms exacts
#                                     + pgvector capte synonymes
# Composants actifs : BM25 MeiliSearch + pgvector MiniLM + RRF
# ============================================================

print('='*65)
print('  COUCHE 2 - Hybrid RAG (S7A)')
print('  BM25 (exact) + pgvector (semantique) + RRF (fusion)')
print('='*65)

QUESTIONS_HYBRID = [
    'solde bancaire par societe',
    'comptes avec leur solde',
    'transactions bancaires recentes',
    'liste les devises disponibles',
    'droits acces utilisateurs',
]

results_hybrid = []
for q in QUESTIONS_HYBRID:
    r = call_api(q)
    results_hybrid.append(r)
    print_result(r)

ok_hybrid = [r for r in results_hybrid if r['success']]
avg_hybrid = round(sum(r['elapsed_ms'] for r in results_hybrid)/len(results_hybrid))
print(f'\n  Succes={len(ok_hybrid)}/{len(results_hybrid)} | Moy={avg_hybrid}ms')
print('  Comparaison BM25 seul vs dense seul vs fusion RRF')


In [ ]:
# ============================================================
# COUCHE 3 : GraphRAG BFS (Sprint 7B)
# Test : questions necess. JOINs complexes -> exploration FK
# Composants actifs : GraphRAG BFS 1223 tables 3394 FK
# ============================================================

print('='*65)
print('  COUCHE 3 - GraphRAG BFS (S7B)')
print('  Exploration FK : 1223 tables - 3394 relations - hops=2')
print('='*65)

QUESTIONS_GRAPHRAG = [
    'financements actifs par banque',
    'comptes BNP avec leur societe',
    'utilisateurs et leurs societes associees',
    'transactions avec le compte associe',
    'financements avec les informations banque',
]

results_graph = []
for q in QUESTIONS_GRAPHRAG:
    r = call_api(q)
    results_graph.append(r)
    print_result(r)

ok_graph = [r for r in results_graph if r['success']]
avg_graph = round(sum(r['elapsed_ms'] for r in results_graph)/len(results_graph))
print(f'\n  Succes={len(ok_graph)}/{len(results_graph)} | Moy={avg_graph}ms')
print('  JOIN paths valides : explorations BFS depuis tables seed')


In [ ]:
# ============================================================
# COUCHE 4 : Corrective RAG - CRAG (Sprint 7C)
# Test : questions ambigues -> reranker qualifie le contexte
# Composants actifs : Cross-encoder score seuil -3.0 + re-query
# ============================================================

print('='*65)
print('  COUCHE 4 - Corrective RAG (S7C)')
print('  Reranker cross-encoder + seuil -3.0 + re-query si mauvais')
print('='*65)

QUESTIONS_CRAG = [
    'informations sur les entites financieres',
    'donnees de tresorerie consolidees',
    'analyse des flux monetaires',
    'statistiques sur les operations bancaires',
    'reporting financier global',
]

results_crag = []
for q in QUESTIONS_CRAG:
    r = call_api(q)
    results_crag.append(r)
    print_result(r)

ok_crag = [r for r in results_crag if r['success']]
avg_crag = round(sum(r['elapsed_ms'] for r in results_crag)/len(results_crag))
print(f'\n  Succes={len(ok_crag)}/{len(results_crag)} | Moy={avg_crag}ms')
print('  CRAG : re-query si reranker score < -3.0')


In [ ]:
# ============================================================
# COUCHE 5 : Direct SQL Bypass (Sprint 8.5 / 8.6)
# Test : questions connues -> bypass total LLM
# Composants actifs : SXA_DIRECT_SQL dict - 3 passes matching
# ============================================================

print('='*65)
print('  COUCHE 5 - Direct SQL Bypass (S8.5/S8.6)')
print('  0 LLM - 0 RAG - ~350ms - 3 passes : exact/synonymes/Jaccard')
print('='*65)

QUESTIONS_DIRECT = [
    'liste les devises',
    'financements actifs',
    'solde bancaire',
    'taux de change',
    'comptes BNP',
    'comptes Societe Generale',
    'utilisateurs bloques',
    'financements clotures',
    'financements BNP',
    'total des transactions par devise en 2024',
    'utilisateurs actifs',
    'financements dont la maturite depasse la duree moyenne des financements BNP',
    'transactions TND superieures a 10000 La Banque Postale 2024',
    'utilisateurs avec acces a plus dune societe',
    'financements par banque et par devise en 2024',
    'financements par banque et par type',
    'total des financements par banque et par devise en 2024',
    'top 10 financements par banque et par type de transaction',
    'liste les banques',
    'liste les societes',
]

results_direct = []
for q in QUESTIONS_DIRECT:
    r = call_api(q)
    results_direct.append(r)
    print_result(r)

ok_direct = [r for r in results_direct if r['success']]
direct_agents = [r for r in results_direct if 'direct' in r.get('agent','')]
avg_direct = round(sum(r['elapsed_ms'] for r in results_direct)/len(results_direct))
print(f'\n  Succes={len(ok_direct)}/{len(results_direct)} | Direct SQL={len(direct_agents)}/{len(results_direct)} | Moy={avg_direct}ms')


In [ ]:
# ============================================================
# COUCHE 6 : Agentic RAG - ReAct (Sprint 8)
# Test : questions inconnues -> boucle Reason-Act-Observe
# Composants actifs : CRAG + LLM boucle max 5 iter
# ============================================================

print('='*65)
print('  COUCHE 6 - Agentic RAG ReAct (S8)')
print('  Questions inconnues -> boucle ReAct max 5 iter')
print('='*65)

QUESTIONS_REACT = [
    'nombre de comptes par devise et par societe',
    'utilisateurs crees apres janvier 2024',
    'financements avec maturite entre 1000 et 2000 jours',
    'liste les options dacces disponibles dans le systeme',
    'groupe de financement et leurs montants totaux',
]

results_react = []
for q in QUESTIONS_REACT:
    r = call_api(q)
    results_react.append(r)
    print_result(r)

ok_react = [r for r in results_react if r['success']]
hall_react = [r for r in results_react if r['is_hallucination']]
avg_react = round(sum(r['elapsed_ms'] for r in results_react)/len(results_react))
print(f'\n  Succes={len(ok_react)}/{len(results_react)} | Hall={len(hall_react)} | Moy={avg_react}ms')


In [ ]:
# ============================================================
# COUCHE 7 : Multi-Agent RAG + Orchestrateur (Sprint 9)
# Test : questions composees ET/VS -> asyncio.gather parallele
# Composants actifs : Orchestrateur + MultiQuery Agent
# ============================================================

print('='*65)
print('  COUCHE 7 - Multi-Agent RAG Orchestrateur (S9)')
print('  Questions ET/VS -> 2 sous-questions paralleles')
print('='*65)

QUESTIONS_MULTI_ET = [
    'utilisateurs actifs et leurs droits dacces',
    'solde des comptes EUR et total des transactions USD',
    'financements actifs et les comptes associes',
    'liste les banques et les devises associees',
]

QUESTIONS_MULTI_VS = [
    'compare les comptes BNP vs les comptes Societe Generale',
    'compare les financements ouverts vs les financements clotures',
    'compare les utilisateurs actifs vs les utilisateurs bloques',
]

print('  -- Questions ET --')
results_et = []
for q in QUESTIONS_MULTI_ET:
    r = call_api(q)
    results_et.append(r)
    print_result(r)

print('  -- Questions VS --')
results_vs = []
for q in QUESTIONS_MULTI_VS:
    r = call_api(q)
    results_vs.append(r)
    print_result(r)

all_multi = results_et + results_vs
ok_multi = [r for r in all_multi if r['success']]
multi_agents = [r for r in all_multi if r.get('agent') == 'multi_query']
avg_multi = round(sum(r['elapsed_ms'] for r in all_multi)/len(all_multi))
print(f'\n  Succes={len(ok_multi)}/{len(all_multi)} | MultiQuery={len(multi_agents)}/{len(all_multi)} | Moy={avg_multi}ms')


In [ ]:
# ============================================================
# COUCHE 8 : Agent Precision (Sprint 10)
# Test : questions complexes multi-dims -> schema structure reel
# Composants actifs : CRAG + _tool_get_table_columns + LLM guide
# ============================================================

print('='*65)
print('  COUCHE 8 - Agent Precision (S10)')
print('  Schema structure reel -> 0 hallucination')
print('='*65)

QUESTIONS_PRECISION = [
    'total des financements par banque et par devise en 2024',
    'total des financements par type et par etat',
    'repartition des financements par banque et par type',
    'nombre de financements par banque et par societe',
    'evolution des financements par banque sur 2024',
]

results_precision = []
for q in QUESTIONS_PRECISION:
    r = call_api(q)
    results_precision.append(r)
    print_result(r)

ok_prec = [r for r in results_precision if r['success']]
hall_prec = [r for r in results_precision if r['is_hallucination']]
prec_agents = [r for r in results_precision if r.get('agent') == 'precision']
avg_prec = round(sum(r['elapsed_ms'] for r in results_precision)/len(results_precision))
print(f'\n  Succes={len(ok_prec)}/{len(results_precision)} | Precision agent={len(prec_agents)} | Hall={len(hall_prec)} | Moy={avg_prec}ms')


In [ ]:
# ============================================================
# RESUME GLOBAL - Toute l'architecture RAG
# ============================================================

print('='*65)
print('  RESUME GLOBAL - Architecture RAG OnePilot')
print('='*65)

layers = [
    ('S7  RAG Classique',  results_rag),
    ('S7A Hybrid RAG',     results_hybrid),
    ('S7B GraphRAG BFS',   results_graph),
    ('S7C Corrective RAG', results_crag),
    ('S8.5 Direct SQL',    results_direct),
    ('S8  Agentic ReAct',  results_react),
    ('S9  Multi-Agent',    all_multi),
    ('S10 Precision',      results_precision),
]

print(f'  {"Sprint":<22} {"Succes":>8} {"Hall":>6} {"Moy ms":>8} {"Agent":>12}')
print(f'  {"-"*22} {"-"*8} {"-"*6} {"-"*8} {"-"*12}')

all_results_flat = []
for name, results in layers:
    if not results: continue
    ok   = sum(1 for r in results if r['success'])
    hall = sum(1 for r in results if r['is_hallucination'])
    avg  = round(sum(r['elapsed_ms'] for r in results)/len(results))
    agents_count = {}
    for r in results:
        a = r.get('agent','?')[:10]
        agents_count[a] = agents_count.get(a,0)+1
    top_agent = max(agents_count, key=agents_count.get) if agents_count else '?'
    pct = ok/len(results)
    print(f'  {name:<22} {ok:>3}/{len(results):<4} ({pct:.0%}) {hall:>3} hall {avg:>6}ms  {top_agent}')
    all_results_flat.extend(results)

print(f'\n  Total questions testees : {len(all_results_flat)}')
total_ok   = sum(1 for r in all_results_flat if r['success'])
total_hall = sum(1 for r in all_results_flat if r['is_hallucination'])
total_avg  = round(sum(r['elapsed_ms'] for r in all_results_flat)/len(all_results_flat))
print(f'  Succes global      : {total_ok}/{len(all_results_flat)} ({total_ok/len(all_results_flat):.0%})')
print(f'  Hallucinations     : {total_hall}/{len(all_results_flat)} ({total_hall/len(all_results_flat):.0%})')
print(f'  Latence moyenne    : {total_avg}ms')


In [ ]:
# ============================================================
# GRAPHIQUES - Evolution par couche RAG
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('OnePilot - Architecture RAG Complete - Benchmark evolutif', fontsize=14, fontweight='bold')

layer_names = ['S7 RAG','S7A Hybrid','S7B Graph','S7C CRAG','S8.5 Direct','S8 ReAct','S9 Multi','S10 Precision']
layer_data  = [results_rag, results_hybrid, results_graph, results_crag,
               results_direct, results_react, all_multi, results_precision]
colors = ['#95a5a6','#3498db','#2ecc71','#e67e22','#1abc9c','#e74c3c','#9b59b6','#f39c12']

def safe_avg(lst): return round(sum(r['elapsed_ms'] for r in lst)/len(lst)) if lst else 0
def hall_rate(lst): return sum(1 for r in lst if r['is_hallucination'])/len(lst)*100 if lst else 0
def suc_rate(lst):  return sum(1 for r in lst if r['success'])/len(lst)*100 if lst else 0

# ── Graphe 1 : Taux de succes par couche ─────────────────────────────────────
rates = [suc_rate(l) for l in layer_data]
bars1 = axes[0,0].bar(range(len(layer_names)), rates, color=colors, alpha=0.85)
axes[0,0].set_xticks(range(len(layer_names)))
axes[0,0].set_xticklabels(layer_names, rotation=30, ha='right', fontsize=9)
axes[0,0].set_title('Taux de succes par couche RAG (%)')
axes[0,0].set_ylabel('%'); axes[0,0].set_ylim(0, 115)
axes[0,0].axhline(80, color='green', linestyle='--', alpha=0.5, label='Objectif 80%')
axes[0,0].legend(fontsize=8)
for bar, val in zip(bars1, rates):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                   f'{val:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Graphe 2 : Taux d'hallucination par couche ───────────────────────────────
hall_rates = [hall_rate(l) for l in layer_data]
bars2 = axes[0,1].bar(range(len(layer_names)), hall_rates, color=colors, alpha=0.85)
axes[0,1].set_xticks(range(len(layer_names)))
axes[0,1].set_xticklabels(layer_names, rotation=30, ha='right', fontsize=9)
axes[0,1].set_title('Taux hallucination par couche RAG (%)')
axes[0,1].set_ylabel('%')
axes[0,1].axhline(10, color='red', linestyle='--', alpha=0.5, label='Seuil max 10%')
axes[0,1].legend(fontsize=8)
for bar, val in zip(bars2, hall_rates):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                   f'{val:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Graphe 3 : Latence par couche ────────────────────────────────────────────
latences = [safe_avg(l) for l in layer_data]
bars3 = axes[1,0].bar(range(len(layer_names)), latences, color=colors, alpha=0.85)
axes[1,0].set_xticks(range(len(layer_names)))
axes[1,0].set_xticklabels(layer_names, rotation=30, ha='right', fontsize=9)
axes[1,0].set_title('Latence moyenne par couche (ms)')
axes[1,0].set_ylabel('ms')
axes[1,0].axhline(500,  color='green',  linestyle='--', alpha=0.5, label='500ms')
axes[1,0].axhline(5000, color='orange', linestyle='--', alpha=0.5, label='5s')
axes[1,0].legend(fontsize=8)
for bar, val in zip(bars3, latences):
    axes[1,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                   f'{val}ms', ha='center', va='bottom', fontsize=8, fontweight='bold')

# ── Graphe 4 : Distribution agents ───────────────────────────────────────────
all_flat = [r for l in layer_data for r in l]
agent_counts = {}
for r in all_flat:
    a = r.get('agent','unknown')
    agent_counts[a] = agent_counts.get(a,0)+1
agent_labels = list(agent_counts.keys())
agent_vals   = list(agent_counts.values())
agent_colors = ['#1abc9c','#f39c12','#e74c3c','#9b59b6','#95a5a6']
bars4 = axes[1,1].bar(range(len(agent_labels)), agent_vals,
                       color=agent_colors[:len(agent_labels)], alpha=0.85, width=0.5)
axes[1,1].set_xticks(range(len(agent_labels)))
axes[1,1].set_xticklabels(agent_labels, rotation=15, ha='right', fontsize=9)
axes[1,1].set_title('Distribution des agents sur toutes les questions')
axes[1,1].set_ylabel('Nb questions')
for bar, val in zip(bars4, agent_vals):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                   str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('benchmark_rag_complet.png', dpi=150, bbox_inches='tight')
plt.close()
print('Graphique sauvegarde : benchmark_rag_complet.png')
